Выделить на GPU массив arr из 10^9 элементов типа float и инициализировать его с помощью ядра следующим образом: arr[i] = sin((i%360)*Pi/180).

Скопировать массив в память центрального процессора и посчитать ошибку err = sum_i(abs(sin((i%360)*Pi/180) - arr[i]))/10^9.

Провести исследование зависимости результата от использования функций: sin, sinf, __sinf. Объяснить результат.

Проверить результат при использовании массива типа double. Сравнить время работы этих функций с разными типами.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

1. Выделяем массив $10^9$ элементов `float` и `double`

In [2]:
%%writefile sin_benchmark.cu
#include <iostream>
#include <cmath>
#include <fstream>
#include <cuda_runtime.h>

const long long N = 1000000000LL; // 10^9
const float PI_F = 3.14159265358979323846f;
const double PI_D = 3.14159265358979323846;

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = call; \
        if (err != cudaSuccess) { \
            std::cerr << "CUDA Error: " << cudaGetErrorString(err) << std::endl; \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

// --- CUDA Ядра ---
__global__ void init_float_sin(float* arr, long long n) {
    long long i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        float angle = (i % 360) * PI_F / 180.0f;
        arr[i] = sin(angle);
    }
}

__global__ void init_float_sinf(float* arr, long long n) {
    long long i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        float angle = (i % 360) * PI_F / 180.0f;
        arr[i] = sinf(angle);
    }
}

__global__ void init_float___sinf(float* arr, long long n) {
    long long i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        float angle = (i % 360) * PI_F / 180.0f;
        arr[i] = __sinf(angle);
    }
}

__global__ void init_double_sin(double* arr, long long n) {
    long long i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        double angle = (i % 360) * PI_D / 180.0;
        arr[i] = sin(angle);
    }
}

// --- Расчет ошибки на CPU ---
double calc_error_float(const float* h_arr) {
    double err = 0.0;
    for (long long i = 0; i < N; i++) {
        float ref = sinf((i % 360) * PI_F / 180.0f);
        err += std::abs(ref - h_arr[i]);
    }
    return err / N;
}

double calc_error_double(const double* h_arr) {
    double err = 0.0;
    for (long long i = 0; i < N; i++) {
        double ref = sin((i % 360) * PI_D / 180.0);
        err += std::abs(ref - h_arr[i]);
    }
    return err / N;
}

int main() {
    int blockSize = 256;
    long long numBlocks = (N + blockSize - 1) / blockSize;

    std::ofstream csv("sin_results.csv");
    csv << "Function,Type,Time_ms,Error\n";

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float ms;

    // --- Тесты для FLOAT ---
    float* d_arr_f;
    float* h_arr_f = new float[N];
    CUDA_CHECK(cudaMalloc(&d_arr_f, N * sizeof(float)));

    // 1. sin (float)
    cudaEventRecord(start);
    init_float_sin<<<numBlocks, blockSize>>>(d_arr_f, N);
    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    cudaMemcpy(h_arr_f, d_arr_f, N * sizeof(float), cudaMemcpyDeviceToHost);
    csv << "sin,float," << ms << "," << calc_error_float(h_arr_f) << "\n";

    // 2. sinf (float)
    cudaEventRecord(start);
    init_float_sinf<<<numBlocks, blockSize>>>(d_arr_f, N);
    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    cudaMemcpy(h_arr_f, d_arr_f, N * sizeof(float), cudaMemcpyDeviceToHost);
    csv << "sinf,float," << ms << "," << calc_error_float(h_arr_f) << "\n";

    // 3. __sinf (float)
    cudaEventRecord(start);
    init_float___sinf<<<numBlocks, blockSize>>>(d_arr_f, N);
    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    cudaMemcpy(h_arr_f, d_arr_f, N * sizeof(float), cudaMemcpyDeviceToHost);
    csv << "__sinf,float," << ms << "," << calc_error_float(h_arr_f) << "\n";

    cudaFree(d_arr_f);
    delete[] h_arr_f;

    // --- Тесты для DOUBLE ---
    double* d_arr_d;
    double* h_arr_d = new double[N];
    cudaError_t err = cudaMalloc(&d_arr_d, N * sizeof(double));
    if (err == cudaSuccess) {
        cudaEventRecord(start);
        init_double_sin<<<numBlocks, blockSize>>>(d_arr_d, N);
        cudaEventRecord(stop); cudaEventSynchronize(stop);
        cudaEventElapsedTime(&ms, start, stop);
        cudaMemcpy(h_arr_d, d_arr_d, N * sizeof(double), cudaMemcpyDeviceToHost);
        csv << "sin,double," << ms << "," << calc_error_double(h_arr_d) << "\n";
        cudaFree(d_arr_d);
        delete[] h_arr_d;
    } else {
        std::cerr << "Не хватило VRAM для double." << std::endl;
    }

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    csv.close();
    std::cout << "Готово! Результаты сохранены в sin_results.csv" << std::endl;
    return 0;
}

Writing sin_benchmark.cu


2. Инициализируем ядрами с функциями `sin`, `sinf`, `__sinf`.


In [3]:
!nvcc -O3 -o sin_benchmark sin_benchmark.cu && ./sin_benchmark

Готово! Результаты сохранены в sin_results.csv


3. Копируем на CPU, считаем среднюю абсолютную ошибку и замеряем время работы ядер через CUDA Events.

In [6]:
df = pd.read_csv('sin_results.csv')

df.columns = ['Функция', 'Тип данных', 'Время (мс)', 'Ошибка']

df['Время (мс)'] = df['Время (мс)'].map(lambda x: f"{x:.2f}")
df['Ошибка'] = df['Ошибка'].map(lambda x: f"{x:.2e}")

display(df)

,Функция,Тип данных,Время (мс),Ошибка
0,sin,float,25.14,8.69e-09
1,sinf,float,31.32,8.69e-09
2,__sinf,float,24.40,1.23e-07
3,sin,double,325.93,8.78e-18
